# House Price Predictor: AWS Machine Learning Basics Lab

BITS Pilani | Professional Certificate in AI/ML

This notebook builds the 9-step pipeline from the Business Problem Statement: generate synthetic property data, upload it to S3, catalogue it with Glue, query it with Athena, register the features in SageMaker Feature Store, train a Random Forest locally, and register the versioned model in SageMaker Model Registry behind a manual approval gate.

No SageMaker training or hosting jobs run here. Everything in this notebook is serverless or API-only, so it works fine even on an account with ML compute quota set to zero.

| Step | Action | Service |
|---|---|---|
| 1 | Generate 500 house records | pandas / NumPy |
| 2 | Upload `houses.csv` to the raw zone | Amazon S3 |
| 3 | Register `houses_raw` external table | AWS Glue Data Catalog |
| 4 | Query average price by bedroom count | Amazon Athena |
| 5 | Ingest 500 feature records | SageMaker Feature Store |
| 6 | Train Random Forest (80/20 split) | scikit-learn, local |
| 7 | Package and upload `model.tar.gz` | Amazon S3 |
| 8 | Write the Model Card | Amazon S3 |
| 9 | Register model as `PendingManualApproval` | SageMaker Model Registry |

Run the cells top to bottom, since later ones depend on variables and resources the earlier ones create. Re-running from the top is fine too, the resource-creation cells already catch "already exists" errors.

## 0. Setup

Establishes the AWS session and derives every resource name from the account ID and
region, so the bucket name is globally unique without manual editing.

In [15]:
import io
import json
import os
import tarfile
import time

import boto3
import numpy as np
import pandas as pd
import sagemaker

# ---- Session and identity -------------------------------------------------
boto_session = boto3.Session()
REGION = boto_session.region_name or "us-east-1"
ACCOUNT_ID = boto_session.client("sts").get_caller_identity()["Account"]

sagemaker_session = sagemaker.Session(boto_session=boto_session)

try:
    ROLE = sagemaker.get_execution_role(sagemaker_session)
except Exception:
    # Not running inside SageMaker Studio / notebook instance.
    # Paste the ARN of a role with SageMaker + S3 + Glue permissions here.
    # ROLE = "arn:aws:iam::{}:role/service-role/AmazonSageMaker-ExecutionRole".format(ACCOUNT_ID)
    ROLE = "arn:aws:iam::{}:role/AmazonSageMaker-ExecutionRole".format(ACCOUNT_ID)

    print("WARNING: could not auto-detect execution role. Using fallback:")
    print("        ", ROLE)

# ---- Resource naming ------------------------------------------------------
LAB_ID = "lab01"
BUCKET = f"houselab-{LAB_ID}-{ACCOUNT_ID}-{REGION}"
PREFIX = f"houselab/{LAB_ID}"

RAW_KEY = f"{PREFIX}/raw/houses.csv"
ATHENA_OUTPUT = f"s3://{BUCKET}/{PREFIX}/athena-results/"
FEATURE_STORE_URI = f"s3://{BUCKET}/{PREFIX}/feature-store"
MODEL_KEY = f"{PREFIX}/model/model.tar.gz"
CARD_KEY = f"{PREFIX}/model/model_card.md"

DATABASE = "houselab_db"
TABLE = "houses_raw"
FEATURE_GROUP = f"house-price-fg-{LAB_ID}"
MODEL_PACKAGE_GROUP = f"house-price-predictor-{LAB_ID}"

# ---- Clients --------------------------------------------------------------
s3 = boto_session.client("s3", region_name=REGION)
glue = boto_session.client("glue", region_name=REGION)
athena = boto_session.client("athena", region_name=REGION)
sm = boto_session.client("sagemaker", region_name=REGION)

print(f"Region      : {REGION}")
print(f"Account     : {ACCOUNT_ID}")
print(f"Bucket      : {BUCKET}")
print(f"Prefix      : {PREFIX}")
print(f"Role        : {ROLE}")

Couldn't call 'get_role' to get Role ARN from role name houselab-student to get Role path.


         arn:aws:iam::381492281291:role/AmazonSageMaker-ExecutionRole
Region      : us-east-1
Account     : 381492281291
Bucket      : houselab-lab01-381492281291-us-east-1
Prefix      : houselab/lab01
Role        : arn:aws:iam::381492281291:role/AmazonSageMaker-ExecutionRole


---
## Step 1 — Generate the dataset

500 synthetic records built from the price formula in the problem statement:

```
price = 150·size_sqft + 20,000·bedrooms − 1,500·age_years
        − 8,000·distance_km + 25,000·has_garage + Normal(0, 30,000)
```

clipped to \$50,000–\$2,000,000 and rounded to the nearest \$1,000.

`np.random.seed(42)` is set before generation so every learner produces an identical
dataset and can compare metrics directly.

In [2]:
np.random.seed(42)

N_RECORDS = 500

size_sqft = np.random.randint(600, 3000, N_RECORDS)        # 600 - 2999
bedrooms = np.random.randint(1, 6, N_RECORDS)              # 1 - 5
age_years = np.random.randint(0, 50, N_RECORDS)            # 0 - 49
distance_km = np.round(np.random.uniform(1.0, 30.0, N_RECORDS), 2)
has_garage = np.random.randint(0, 2, N_RECORDS)            # 0 or 1

noise = np.random.normal(loc=0, scale=30_000, size=N_RECORDS)

price_raw = (
    150 * size_sqft
    + 20_000 * bedrooms
    - 1_500 * age_years
    - 8_000 * distance_km
    + 25_000 * has_garage
    + noise
)

price_usd = np.clip(price_raw, 50_000, 2_000_000)
price_usd = (np.round(price_usd / 1000) * 1000).astype(int)

df = pd.DataFrame({
    "size_sqft": size_sqft,
    "bedrooms": bedrooms,
    "age_years": age_years,
    "distance_km": distance_km,
    "has_garage": has_garage,
    "price_usd": price_usd,
})

print(f"Generated {len(df)} records\n")
print(df.head(10).to_string(index=False))
print("\nSummary statistics:")
print(df.describe().round(1).to_string())

n_floored = int((price_raw < 50_000).sum())
print(f"\nRecords clipped at the $50,000 floor: {n_floored} ({n_floored / N_RECORDS:.1%})")

Generated 500 records

 size_sqft  bedrooms  age_years  distance_km  has_garage  price_usd
      1460         5         27         4.74           0     216000
      1894         1         49        10.72           1     107000
      1730         4         20        22.56           0     147000
      1695         1         48         5.66           1     192000
      2238         1          6        24.72           0      98000
      2769         5         16        25.13           0     334000
      1066         4         19        15.72           1     161000
      1838         4         40         1.19           1     289000
       930         4         48         9.32           0     108000
      2082         3         19        18.89           1     200000

Summary statistics:
       size_sqft  bedrooms  age_years  distance_km  has_garage  price_usd
count      500.0     500.0      500.0        500.0       500.0      500.0
mean      1827.2       3.0       25.4         15.5         0

**Observation worth recording.** About 17% of records land below the \$50,000 floor
before clipping and get pinned to exactly \$50,000. This is *left censoring*: for those
properties the target no longer follows the linear formula, so no model can recover the
true relationship in that region. It sets a practical ceiling on achievable R² and is
the main reason the score lands near 0.91 rather than close to 1.0 despite the data
being generated from a known linear equation. In a real valuation system the equivalent
problem is a price floor imposed by land value.

---
## Step 2 — Upload to Amazon S3

Creates the bucket and writes the DataFrame as CSV directly from memory using
`put_object()` — no local file is written.

**Region quirk:** `us-east-1` is the S3 default and *rejects* a
`CreateBucketConfiguration` block. Every other region *requires* one. The code branches
on this; getting it wrong is the single most common failure in this step.

In [3]:
# ---- Create the bucket ----------------------------------------------------
try:
    if REGION == "us-east-1":
        s3.create_bucket(Bucket=BUCKET)
    else:
        s3.create_bucket(
            Bucket=BUCKET,
            CreateBucketConfiguration={"LocationConstraint": REGION},
        )
    print(f"Created bucket: {BUCKET}")
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f"Bucket already exists and is owned by you: {BUCKET}")
except s3.exceptions.BucketAlreadyExists:
    print(f"Bucket name is taken globally: {BUCKET}")
    print("Change LAB_ID in the Setup cell and re-run.")

# ---- Upload the CSV -------------------------------------------------------
csv_buffer = io.StringIO()
df.to_csv(csv_buffer, index=False)

s3.put_object(
    Bucket=BUCKET,
    Key=RAW_KEY,
    Body=csv_buffer.getvalue().encode("utf-8"),
    ContentType="text/csv",
)

head = s3.head_object(Bucket=BUCKET, Key=RAW_KEY)
print(f"\nUploaded  : s3://{BUCKET}/{RAW_KEY}")
print(f"Size      : {head['ContentLength']:,} bytes")
print(f"S3 URI    : s3://{BUCKET}/{RAW_KEY}")

Created bucket: houselab-lab01-381492281291-us-east-1

Uploaded  : s3://houselab-lab01-381492281291-us-east-1/houselab/lab01/raw/houses.csv
Size      : 12,063 bytes
S3 URI    : s3://houselab-lab01-381492281291-us-east-1/houselab/lab01/raw/houses.csv


**Vocabulary check (learning objective).**

| Term | Meaning here |
|---|---|
| **Bucket** | `houselab-lab01-…` — the globally unique top-level container |
| **Key** | `houselab/lab01/raw/houses.csv` — the full path within the bucket |
| **Object** | The CSV bytes plus their metadata (size, content type, ETag) |
| **S3 URI** | `s3://bucket/key` — the address AWS services use to reference the object |

S3 has no real directories. The slashes in the key are just characters; the console
renders them as folders as a convenience.

---
## Step 3 — Register the table in the Glue Data Catalog

The Glue Data Catalog holds *metadata only* — it stores the schema and the S3 location,
never a copy of the data. Athena reads this metadata to learn how to parse the file.

**Two decisions to note:**

1. **All columns are declared `string`.** `OpenCSVSerde` reads every CSV field as text
   regardless of the declared type. Declaring `int` here would produce silent type
   errors at query time, so we declare `string` and cast in SQL instead — which is
   exactly why Step 4 needs `CAST`.
2. **`skip.header.line.count = 1`** stops the header row being returned as a data row.

In [4]:
# ---- Create the database (a namespace for tables) -------------------------
try:
    glue.create_database(
        DatabaseInput={
            "Name": DATABASE,
            "Description": "House price lab - raw zone catalogue",
        }
    )
    print(f"Created Glue database: {DATABASE}")
except glue.exceptions.AlreadyExistsException:
    print(f"Glue database already exists: {DATABASE}")

# ---- Register the external table ------------------------------------------
COLUMNS = ["size_sqft", "bedrooms", "age_years",
           "distance_km", "has_garage", "price_usd"]

table_input = {
    "Name": TABLE,
    "TableType": "EXTERNAL_TABLE",
    "Parameters": {
        "classification": "csv",
        "skip.header.line.count": "1",
    },
    "StorageDescriptor": {
        "Columns": [{"Name": c, "Type": "string"} for c in COLUMNS],
        # Points at the FOLDER, not the file - Athena scans every object under it.
        "Location": f"s3://{BUCKET}/{PREFIX}/raw/",
        "InputFormat": "org.apache.hadoop.mapred.TextInputFormat",
        "OutputFormat": "org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat",
        "SerdeInfo": {
            "SerializationLibrary": "org.apache.hadoop.hive.serde2.OpenCSVSerde",
            "Parameters": {
                "separatorChar": ",",
                "quoteChar": '"',
                "escapeChar": "\\",
            },
        },
    },
}

try:
    glue.create_table(DatabaseName=DATABASE, TableInput=table_input)
    print(f"Created Glue table: {DATABASE}.{TABLE}")
except glue.exceptions.AlreadyExistsException:
    glue.update_table(DatabaseName=DATABASE, TableInput=table_input)
    print(f"Glue table already existed - updated: {DATABASE}.{TABLE}")

# ---- Verify ---------------------------------------------------------------
meta = glue.get_table(DatabaseName=DATABASE, Name=TABLE)["Table"]
print(f"\nTable type : {meta['TableType']}")
print(f"Location   : {meta['StorageDescriptor']['Location']}")
print(f"SerDe      : {meta['StorageDescriptor']['SerdeInfo']['SerializationLibrary'].split('.')[-1]}")
print("Columns    :", ", ".join(c["Name"] for c in meta["StorageDescriptor"]["Columns"]))

Glue database already exists: houselab_db
Glue table already existed - updated: houselab_db.houses_raw

Table type : EXTERNAL_TABLE
Location   : s3://houselab-lab01-381492281291-us-east-1/houselab/lab01/raw/
SerDe      : OpenCSVSerde
Columns    : size_sqft, bedrooms, age_years, distance_km, has_garage, price_usd


**External vs. regular table (learning objective).**

An **external table** describes data that lives outside the query engine — here, a CSV
sitting in S3. Glue owns only the schema. `DROP TABLE` deletes the metadata and leaves
the CSV untouched, and other tools can read the same file simultaneously.

A **regular (managed) table** means the engine owns the storage. Dropping it deletes the
underlying data.

The **SerDe** (Serializer/Deserializer) is the plug-in that translates raw bytes into
rows and columns. `OpenCSVSerde` handles quoted fields and embedded commas correctly,
which the older `LazySimpleSerDe` does not.

---
## Step 4 — Query with Amazon Athena

Athena runs Presto SQL directly against S3 — serverless, no cluster to provision. The
boto3 API is **asynchronous**: submit, poll, then fetch. The three calls map to the
three learning objectives for this service.

`CAST(price_usd AS DOUBLE)` is mandatory: the Glue schema declares the column as
`string`, and `AVG()` cannot aggregate text.

In [5]:
QUERY = f"""
SELECT
    CAST(bedrooms AS INTEGER)                        AS bedrooms,
    COUNT(*)                                         AS num_houses,
    ROUND(AVG(CAST(price_usd AS DOUBLE)), 0)         AS avg_price_usd,
    ROUND(AVG(CAST(size_sqft AS DOUBLE)), 0)         AS avg_size_sqft
FROM {DATABASE}.{TABLE}
GROUP BY CAST(bedrooms AS INTEGER)
ORDER BY bedrooms
"""

# ---- 1. Submit (returns immediately with an execution ID) ------------------
response = athena.start_query_execution(
    QueryString=QUERY,
    QueryExecutionContext={"Database": DATABASE},
    ResultConfiguration={"OutputLocation": ATHENA_OUTPUT},
)
query_id = response["QueryExecutionId"]
print(f"Submitted query: {query_id}")

# ---- 2. Poll until the query reaches a terminal state ----------------------
TERMINAL = {"SUCCEEDED", "FAILED", "CANCELLED"}
for attempt in range(60):
    execution = athena.get_query_execution(QueryExecutionId=query_id)
    state = execution["QueryExecution"]["Status"]["State"]
    if state in TERMINAL:
        break
    time.sleep(2)

print(f"Final state    : {state}")

if state != "SUCCEEDED":
    reason = execution["QueryExecution"]["Status"].get("StateChangeReason", "unknown")
    raise RuntimeError(f"Athena query {state}: {reason}")

stats = execution["QueryExecution"]["Statistics"]
print(f"Data scanned   : {stats['DataScannedInBytes']:,} bytes")
print(f"Runtime        : {stats['EngineExecutionTimeInMillis']} ms")

# ---- 3. Fetch and parse results into a DataFrame ---------------------------
results = athena.get_query_results(QueryExecutionId=query_id)
rows = results["ResultSet"]["Rows"]

header = [c["VarCharValue"] for c in rows[0]["Data"]]
data = [[c.get("VarCharValue") for c in r["Data"]] for r in rows[1:]]

athena_df = pd.DataFrame(data, columns=header)
athena_df = athena_df.apply(pd.to_numeric)

print("\nAverage price by bedroom count:")
print(athena_df.to_string(index=False))

Submitted query: 34657188-af85-45ce-831e-7901be8f309a
Final state    : SUCCEEDED
Data scanned   : 12,063 bytes
Runtime        : 508 ms

Average price by bedroom count:
 bedrooms  num_houses  avg_price_usd  avg_size_sqft
        1         107       162692.0         1857.0
        2          94       170394.0         1746.0
        3         100       195840.0         1828.0
        4          93       209462.0         1811.0
        5         106       237217.0         1883.0


Average price rises monotonically with bedroom count, which is the expected direction —
though the gap is wider than the \$20,000-per-bedroom coefficient alone would explain.
That is a **confounding effect**: `bedrooms` and `size_sqft` are independently generated
here, but the floor-clipping removes disproportionately many cheap (small, few-bedroom)
houses, inflating the low-bedroom averages' apparent spread. Athena tells you *what*
the aggregate looks like; only the model in Step 6 isolates each feature's true
contribution.

---
## Step 5 — SageMaker Feature Store

Feature Store solves **training/serving skew**: when the data science team computes a
feature one way in a notebook and the engineering team reimplements it slightly
differently in production, the model silently degrades. A central store means both
teams read the identical, versioned definition.

Feature Store accepts only three types — `String`, `Integral`, `Fractional` — and
requires two special columns:

- **Record identifier** — the primary key (`record_id`)
- **Event time** — when the feature values were valid (`event_time`), as an epoch
  Fractional or an ISO-8601 String

In [6]:
from sagemaker.feature_store.feature_group import FeatureGroup

# ---- Prepare the DataFrame ------------------------------------------------
fs_df = df.copy()
fs_df["record_id"] = fs_df.index.astype(str)          # String identifier
fs_df["event_time"] = float(round(time.time(), 3))    # Fractional epoch seconds

# Numeric columns must be an explicit float dtype -> Fractional.
for col in ["size_sqft", "bedrooms", "age_years",
            "distance_km", "has_garage", "price_usd"]:
    fs_df[col] = fs_df[col].astype("float64")

print(fs_df.dtypes.to_string())
print(f"\nRows to ingest: {len(fs_df)}")
print(repr(ROLE))


# ---- Create the Feature Group ---------------------------------------------
feature_group = FeatureGroup(name=FEATURE_GROUP, sagemaker_session=sagemaker_session)
feature_group.load_feature_definitions(data_frame=fs_df)

try:
    feature_group.create(
        s3_uri=FEATURE_STORE_URI,
        record_identifier_name="record_id",
        event_time_feature_name="event_time",
        role_arn=ROLE,
        enable_online_store=True,          # online + offline
        description="Property features for the house price predictor lab",
    )
    print(f"\nCreating feature group: {FEATURE_GROUP}")
except Exception as exc:
    if "ResourceInUse" in str(exc):
        print(f"\nFeature group already exists: {FEATURE_GROUP}")
    else:
        raise

# ---- Poll until Created ---------------------------------------------------
for attempt in range(60):
    status = feature_group.describe()["FeatureGroupStatus"]
    if status != "Creating":
        break
    time.sleep(5)

print(f"Feature group status: {status}")
if status != "Created":
    raise RuntimeError(f"Feature group did not reach Created state: {status}")

size_sqft      float64
bedrooms       float64
age_years      float64
distance_km    float64
has_garage     float64
price_usd      float64
record_id       object
event_time     float64

Rows to ingest: 500
'arn:aws:iam::381492281291:role/AmazonSageMaker-ExecutionRole'

Feature group already exists: house-price-fg-lab01
Feature group status: Created


In [7]:
# ---- Ingest the records ---------------------------------------------------
feature_group.ingest(data_frame=fs_df, max_workers=1, wait=True)
print(f"Ingested {len(fs_df)} records into {FEATURE_GROUP}")

# ---- Verify by reading one record back from the ONLINE store --------------
featurestore_runtime = boto_session.client(
    "sagemaker-featurestore-runtime", region_name=REGION
)

record = featurestore_runtime.get_record(
    FeatureGroupName=FEATURE_GROUP,
    RecordIdentifierValueAsString="0",
)

print("\nRecord 0 retrieved from the online store:")
for feature in record["Record"]:
    print(f"  {feature['FeatureName']:<14} = {feature['ValueAsString']}")

Ingested 500 records into house-price-fg-lab01

Record 0 retrieved from the online store:
  size_sqft      = 1460.0
  bedrooms       = 5.0
  age_years      = 27.0
  distance_km    = 4.74
  has_garage     = 0.0
  price_usd      = 216000.0
  record_id      = 0
  event_time     = 1786584293.79


**Online vs. offline store (learning objective).**

| | Online store | Offline store |
|---|---|---|
| Backing | Low-latency key-value store | Parquet files in S3 |
| Latency | Single-digit milliseconds | Seconds to minutes |
| Returns | Only the **latest** value per record | **Full history** of every write |
| Used for | Real-time inference — fetch features for one property at request time | Training and batch scoring — read the whole dataset |
| Cost | Per read/write + storage | S3 storage only |

Both are written from a single `ingest()` call, which is precisely the point: one write
path, two read paths, no chance of the two drifting apart. Note the offline store lags —
records appear in S3 within a few minutes, so an immediate query there may return empty
while the online read above succeeds instantly.

---
## Step 6 — Train the Random Forest

Trained locally with scikit-learn inside this notebook — no SageMaker training job, so
no ML compute quota is consumed.

A Random Forest builds 100 decision trees, each on a random bootstrap sample of rows and
a random subset of features at every split. Predictions are the average across all
trees. Because the trees' errors are largely uncorrelated, averaging cancels much of the
variance that makes a single deep tree unreliable.

In [8]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

FEATURES = ["size_sqft", "bedrooms", "age_years", "distance_km", "has_garage"]
TARGET = "price_usd"

X = df[FEATURES]
y = df[TARGET]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training rows  : {len(X_train)}")
print(f"Validation rows: {len(X_val)}")

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)

mae = mean_absolute_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print(f"\nMAE : ${mae:,.0f}")
print(f"R2  : {r2:.4f}")
print(f"\nMean actual price: ${y_val.mean():,.0f}")
print(f"MAE as % of mean : {mae / y_val.mean():.1%}")

Training rows  : 400
Validation rows: 100

MAE : $26,019
R2  : 0.9131

Mean actual price: $202,150
MAE as % of mean : 12.9%


In [9]:
# ---- Feature importance ---------------------------------------------------
importance = (
    pd.Series(model.feature_importances_, index=FEATURES)
    .sort_values(ascending=False)
)

print("Feature importance (Gini / impurity reduction):\n")
for name, value in importance.items():
    bar = "#" * int(value * 50)
    print(f"  {name:<12} {value:.4f}  {bar}")

Feature importance (Gini / impurity reduction):

  size_sqft    0.6387  ###############################
  distance_km  0.2730  #############
  age_years    0.0433  ##
  bedrooms     0.0364  #
  has_garage   0.0087  


### Interpreting the results

**MAE ≈ \$26,000** against a noise standard deviation of \$30,000. For a normal
distribution, the mean absolute deviation is about 0.8σ ≈ \$24,000 — so the model is
operating close to the *irreducible* error floor. Most of the remaining gap comes from
the clipped records, not from underfitting. Nearly all of the error is noise the model
could never have learned, which is the correct outcome.

**R² ≈ 0.91** means the model explains 91% of price variance.

**Feature importance departs from the problem statement's prediction.** Section 4.4
expects `has_garage` fourth and `bedrooms` fifth; the trained model consistently reverses
them. The prediction there was made by ranking raw coefficients, but importance depends
on a feature's *total range of effect*, not its coefficient:

| Feature | Coefficient | Range | Total price swing |
|---|---|---|---|
| `size_sqft` | 150/sqft | 600–2999 | **~\$359,000** |
| `distance_km` | −8,000/km | 1–30 | **~\$232,000** |
| `age_years` | −1,500/yr | 0–49 | **~\$74,000** |
| `bedrooms` | 20,000/bed | 1–5 | **~\$80,000** |
| `has_garage` | 25,000 flat | 0–1 | **~\$25,000** |

Ranked by swing, `bedrooms` (\$80k) clearly beats `has_garage` (\$25k) — which is what
the model finds. `age_years` still edges ahead of `bedrooms` in the fitted importance
despite a slightly smaller swing, because it is continuous with 50 distinct values and
therefore offers the trees far more places to split than a 5-level integer.

**Business takeaway:** the model is learning the genuine generative signal, not
artefacts. Size and location dominate, matching real-estate intuition — which is the
sanity check Section 4.4 exists to perform.

---
## Step 7 — Package the model artifact

SageMaker expects model artifacts as a **gzipped tarball**, with the serialised model at
the archive root. `arcname="model.joblib"` strips the local directory path — without it
the file lands nested inside the tar and the inference container cannot find it.

In [10]:
import joblib

os.makedirs("model_artifacts", exist_ok=True)
local_model_path = "model_artifacts/model.joblib"
joblib.dump(model, local_model_path)

tar_path = "model_artifacts/model.tar.gz"
with tarfile.open(tar_path, "w:gz") as tar:
    tar.add(local_model_path, arcname="model.joblib")   # arcname flattens the path

with open(tar_path, "rb") as fh:
    s3.put_object(Bucket=BUCKET, Key=MODEL_KEY, Body=fh.read())

MODEL_S3_URI = f"s3://{BUCKET}/{MODEL_KEY}"
size = s3.head_object(Bucket=BUCKET, Key=MODEL_KEY)["ContentLength"]

print(f"Local artifact : {tar_path} ({os.path.getsize(tar_path):,} bytes)")
print(f"Uploaded to    : {MODEL_S3_URI}")
print(f"S3 object size : {size:,} bytes")

with tarfile.open(tar_path, "r:gz") as tar:
    print(f"Archive contents: {tar.getnames()}")

Local artifact : model_artifacts/model.tar.gz (538,260 bytes)
Uploaded to    : s3://houselab-lab01-381492281291-us-east-1/houselab/lab01/model/model.tar.gz
S3 object size : 538,260 bytes
Archive contents: ['model.joblib']


---
## Step 8 — Write the Model Card

The Model Card is the governance artefact: it records what the model does, how it
performs, what it was trained on, and where it must not be used. It is what an auditor
or a downstream team reads before trusting a prediction — and it satisfies the executive
mandate's "auditable" requirement.

In [11]:
model_card = f"""# Model Card — House Price Predictor

## Model Overview

| Field | Value |
|---|---|
| Model name | House Price Predictor |
| Version | 1.0 |
| Owner | PropSmart Realty Ltd. — Data Science |
| Date registered | {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())} |
| Model type | Supervised regression |
| Algorithm | Random Forest Regressor (scikit-learn) |
| Hyperparameters | n_estimators=100, random_state=42 |
| Artifact | {MODEL_S3_URI} |

## Intended Use

**Primary use case.** Generate instant indicative sale price estimates for residential
properties on the PropSmart customer portal, reducing enquiry-to-quote time from
approximately 3 days to under 30 seconds.

**Intended users.** Portal visitors (buyers and sellers) and internal estate agents
performing preliminary comparable analysis.

**Out of scope.** This model must NOT be used as the sole basis for mortgage lending
decisions, formal valuations, insurance underwriting, or tax assessment. It produces an
indicative estimate, not a certified appraisal.

## Training Data

| Field | Value |
|---|---|
| Source | Synthetic dataset generated with NumPy (seed=42) |
| Records | {N_RECORDS} |
| Train / validation split | 80% ({len(X_train)} rows) / 20% ({len(X_val)} rows) |
| Catalogued at | s3://{BUCKET}/{RAW_KEY} |
| Feature Group | {FEATURE_GROUP} |

### Features

| Feature | Type | Range | Description |
|---|---|---|---|
| size_sqft | integer | 600 – 2999 | Floor area in square feet |
| bedrooms | integer | 1 – 5 | Number of bedrooms |
| age_years | integer | 0 – 49 | Age of the property in years |
| distance_km | float | 1.0 – 30.0 | Distance from city centre |
| has_garage | binary | 0 or 1 | Garage present |

**Target:** `price_usd` — sale price in USD.

## Performance

| Metric | Value |
|---|---|
| MAE (validation) | ${mae:,.0f} |
| R-squared (validation) | {r2:.4f} |
| MAE as % of mean price | {mae / y_val.mean():.1%} |

### Feature Importance

| Rank | Feature | Importance |
|---|---|---|
""" + "\n".join(
    f"| {i} | {name} | {value:.4f} |"
    for i, (name, value) in enumerate(importance.items(), start=1)
) + f"""

Size and distance from the city centre dominate, consistent with the known data
generating process and with real-estate domain intuition.

## Limitations and Ethical Considerations

- **Synthetic training data.** The model has never seen a real transaction. Performance
  on production data is unknown and must be re-validated before any deployment.
- **Left censoring.** {n_floored} of {N_RECORDS} records ({n_floored / N_RECORDS:.0%}) were
  clipped at the $50,000 floor. Estimates for low-value properties are unreliable.
- **No location feature beyond distance.** Neighbourhood, school catchment, and
  amenities are unmodelled, and these are among the strongest real price drivers.
- **Extrapolation.** Tree ensembles cannot extrapolate. Properties outside the trained
  ranges (e.g. above 2,999 sqft) will receive predictions capped at the boundary values.
- **Fairness.** No protected or proxy attributes are used. Should postcode-level features
  be added later, they must be audited for correlation with protected characteristics,
  since location data can encode historical discrimination.
- **No drift monitoring.** Out of scope for this lab. Production deployment requires
  monitoring before go-live.

## Approval

Registered to Model Package Group `{MODEL_PACKAGE_GROUP}` with status
**PendingManualApproval**. A human reviewer must approve this version before it can be
promoted. No automatic path to production exists.
"""

s3.put_object(
    Bucket=BUCKET,
    Key=CARD_KEY,
    Body=model_card.encode("utf-8"),
    ContentType="text/markdown",
)

print(f"Model card written to: s3://{BUCKET}/{CARD_KEY}")
print(f"Length: {len(model_card):,} characters\n")
print(model_card[:1200] + "\n...")

Model card written to: s3://houselab-lab01-381492281291-us-east-1/houselab/lab01/model/model_card.md
Length: 3,555 characters

# Model Card — House Price Predictor

## Model Overview

| Field | Value |
|---|---|
| Model name | House Price Predictor |
| Version | 1.0 |
| Owner | PropSmart Realty Ltd. — Data Science |
| Date registered | 2026-08-13 01:26:36 UTC |
| Model type | Supervised regression |
| Algorithm | Random Forest Regressor (scikit-learn) |
| Hyperparameters | n_estimators=100, random_state=42 |
| Artifact | s3://houselab-lab01-381492281291-us-east-1/houselab/lab01/model/model.tar.gz |

## Intended Use

**Primary use case.** Generate instant indicative sale price estimates for residential
properties on the PropSmart customer portal, reducing enquiry-to-quote time from
approximately 3 days to under 30 seconds.

**Intended users.** Portal visitors (buyers and sellers) and internal estate agents
performing preliminary comparable analysis.

**Out of scope.** This model must NO

---
## Step 9 — Register in the SageMaker Model Registry

The Model Registry versions models and enforces an approval gate. Two objects:

- **Model Package Group** — the container holding every version of one logical model
- **Model Package** — a single immutable version

The **container image URI** in the inference specification tells SageMaker which Docker
image can serve this artifact. We never launch it in this lab, but the registry requires
it so any future deployment is fully self-describing.

**The parameter is `ModelApprovalStatus`, not `ApprovalStatus`.** boto3 raises
`ParamValidationError` on the wrong name — the error listed in Section 8 of the problem
statement.

In [12]:
# ---- Model Package Group --------------------------------------------------
try:
    sm.create_model_package_group(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP,
        ModelPackageGroupDescription="Versioned house price regression models",
    )
    print(f"Created model package group: {MODEL_PACKAGE_GROUP}")
except (sm.exceptions.ResourceInUse, sm.exceptions.ClientError) as exc:
    if "already exists" in str(exc) or isinstance(exc, sm.exceptions.ResourceInUse):
        print(f"Model package group already exists: {MODEL_PACKAGE_GROUP}")
    else:
        raise

# ---- Resolve the managed scikit-learn inference image ---------------------
image_uri = sagemaker.image_uris.retrieve(
    framework="sklearn",
    region=REGION,
    version="1.2-1",
    py_version="py3",
    instance_type="ml.m5.large",
    image_scope="inference",
)
print(f"\nInference image: {image_uri}")

# ---- Register the model package -------------------------------------------
response = sm.create_model_package(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    ModelPackageDescription="Random Forest, 100 trees, trained on 500 synthetic records",
    InferenceSpecification={
        "Containers": [{
            "Image": image_uri,
            "ModelDataUrl": MODEL_S3_URI,
        }],
        "SupportedContentTypes": ["text/csv"],
        "SupportedResponseMIMETypes": ["text/csv"],
        "SupportedRealtimeInferenceInstanceTypes": ["ml.m5.large"],
        "SupportedTransformInstanceTypes": ["ml.m5.large"],
    },
    ModelApprovalStatus="PendingManualApproval",   # NOT "ApprovalStatus"
    CustomerMetadataProperties={
        "mae": f"{mae:.2f}",
        "r2": f"{r2:.4f}",
        "algorithm": "RandomForestRegressor",
        "n_estimators": "100",
        "training_rows": str(len(X_train)),
        "model_card": f"s3://{BUCKET}/{CARD_KEY}",
    },
)

MODEL_PACKAGE_ARN = response["ModelPackageArn"]
print(f"\nRegistered: {MODEL_PACKAGE_ARN}")

described = sm.describe_model_package(ModelPackageName=MODEL_PACKAGE_ARN)
print(f"Version   : {described['ModelPackageVersion']}")
print(f"Status    : {described['ModelPackageStatus']}")
print(f"Approval  : {described['ModelApprovalStatus']}")

Model package group already exists: house-price-predictor-lab01

Inference image: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3

Registered: arn:aws:sagemaker:us-east-1:381492281291:model-package/house-price-predictor-lab01/6
Version   : 6
Status    : Completed
Approval  : PendingManualApproval


The model now sits in `PendingManualApproval`. To promote it, a reviewer calls
`update_model_package(ModelPackageArn=..., ModelApprovalStatus="Approved")` or clicks
Approve in SageMaker Studio. **Leave it pending** — the gate being closed is the
deliverable, and Section 7.2 checks for exactly this state.

---
## Verification — Success Criteria

Checks all seven criteria from Section 7.2 of the problem statement.

In [13]:
checks = []

# 1. Bucket exists
try:
    s3.head_bucket(Bucket=BUCKET)
    checks.append(("S3 bucket created", True, BUCKET))
except Exception as exc:
    checks.append(("S3 bucket created", False, str(exc)))

# 2. Raw CSV present
try:
    obj = s3.head_object(Bucket=BUCKET, Key=RAW_KEY)
    checks.append(("Raw CSV uploaded", True, f"{obj['ContentLength']:,} bytes"))
except Exception as exc:
    checks.append(("Raw CSV uploaded", False, str(exc)))

# 3. Glue table registered
try:
    tbl = glue.get_table(DatabaseName=DATABASE, Name=TABLE)["Table"]
    checks.append(("Glue table registered", True, f"{DATABASE}.{tbl['Name']}"))
except Exception as exc:
    checks.append(("Glue table registered", False, str(exc)))

# 4. Athena query succeeded
athena_state = athena.get_query_execution(
    QueryExecutionId=query_id
)["QueryExecution"]["Status"]["State"]
checks.append(("Athena query succeeded", athena_state == "SUCCEEDED",
               f"state={athena_state}, {len(athena_df)} groups"))

# 5. Feature group ready with 500 records
try:
    fg_status = feature_group.describe()["FeatureGroupStatus"]
    checks.append(("Feature Group ready", fg_status == "Created",
                   f"status={fg_status}, {len(fs_df)} records ingested"))
except Exception as exc:
    checks.append(("Feature Group ready", False, str(exc)))

# 6. Model artifact in S3
try:
    obj = s3.head_object(Bucket=BUCKET, Key=MODEL_KEY)
    checks.append(("Model artifact in S3", True, f"{obj['ContentLength']:,} bytes"))
except Exception as exc:
    checks.append(("Model artifact in S3", False, str(exc)))

# 7. Model registered and pending approval
try:
    pkg = sm.describe_model_package(ModelPackageName=MODEL_PACKAGE_ARN)
    ok = pkg["ModelApprovalStatus"] == "PendingManualApproval"
    checks.append(("Model registered (pending)", ok,
                   f"v{pkg['ModelPackageVersion']}, {pkg['ModelApprovalStatus']}"))
except Exception as exc:
    checks.append(("Model registered (pending)", False, str(exc)))

# ---- Report ---------------------------------------------------------------
print("=" * 78)
print("SUCCESS CRITERIA")
print("=" * 78)
for name, passed, detail in checks:
    print(f"[{'PASS' if passed else 'FAIL'}]  {name:<28} {detail}")
print("=" * 78)

passed_count = sum(1 for _, p, _ in checks if p)
print(f"{passed_count} / {len(checks)} criteria met")

# Bonus: model card presence
try:
    s3.head_object(Bucket=BUCKET, Key=CARD_KEY)
    print(f"\nModel card: s3://{BUCKET}/{CARD_KEY}")
except Exception:
    print("\nModel card missing - re-run Step 8")

SUCCESS CRITERIA
[PASS]  S3 bucket created            houselab-lab01-381492281291-us-east-1
[PASS]  Raw CSV uploaded             12,063 bytes
[PASS]  Glue table registered        houselab_db.houses_raw
[PASS]  Athena query succeeded       state=SUCCEEDED, 5 groups
[PASS]  Feature Group ready          status=Created, 500 records ingested
[PASS]  Model artifact in S3         538,260 bytes
[PASS]  Model registered (pending)   v6, PendingManualApproval
7 / 7 criteria met

Model card: s3://houselab-lab01-381492281291-us-east-1/houselab/lab01/model/model_card.md


---
## Cleanup

**Run this after capturing your screenshots for submission.** Deletion is irreversible
and will invalidate the console evidence the success criteria ask for.

The Feature Group's online store and S3 storage are the only components with meaningful
ongoing cost; Glue metadata is free.

In [14]:
RUN_CLEANUP = False   # set to True and re-run this cell to delete everything

if not RUN_CLEANUP:
    print("Cleanup skipped. Set RUN_CLEANUP = True to delete all lab resources.")
else:
    # 1. Model packages, then the group (versions must go first)
    try:
        versions = sm.list_model_packages(
            ModelPackageGroupName=MODEL_PACKAGE_GROUP
        )["ModelPackageSummaryList"]
        for v in versions:
            sm.delete_model_package(ModelPackageName=v["ModelPackageArn"])
            print(f"Deleted model package v{v['ModelPackageVersion']}")
        sm.delete_model_package_group(ModelPackageGroupName=MODEL_PACKAGE_GROUP)
        print(f"Deleted model package group: {MODEL_PACKAGE_GROUP}")
    except Exception as exc:
        print(f"Model registry cleanup: {exc}")

    # 2. Feature group
    try:
        feature_group.delete()
        print(f"Deleting feature group: {FEATURE_GROUP}")
    except Exception as exc:
        print(f"Feature group cleanup: {exc}")

    # 3. Glue table and database (no charge, but keeps the account tidy)
    try:
        glue.delete_table(DatabaseName=DATABASE, Name=TABLE)
        glue.delete_database(Name=DATABASE)
        print(f"Deleted Glue database: {DATABASE}")
    except Exception as exc:
        print(f"Glue cleanup: {exc}")

    # 4. Every object under the prefix, then the bucket
    try:
        paginator = s3.get_paginator("list_objects_v2")
        deleted = 0
        for page in paginator.paginate(Bucket=BUCKET):
            keys = [{"Key": o["Key"]} for o in page.get("Contents", [])]
            if keys:
                s3.delete_objects(Bucket=BUCKET, Delete={"Objects": keys})
                deleted += len(keys)
        print(f"Deleted {deleted} S3 objects")
        s3.delete_bucket(Bucket=BUCKET)
        print(f"Deleted bucket: {BUCKET}")
    except Exception as exc:
        print(f"S3 cleanup: {exc}")

    print("\nCleanup complete.")

Cleanup skipped. Set RUN_CLEANUP = True to delete all lab resources.


---
## Summary

| Deliverable | Location |
|---|---|
| S3 bucket | `houselab-lab01-{account}-{region}` |
| Raw dataset | `houselab/lab01/raw/houses.csv` |
| Glue table | `houselab_db.houses_raw` |
| Athena result | `houselab/lab01/athena-results/` |
| Feature Group | `house-price-fg-lab01` (500 records) |
| Model artifact | `houselab/lab01/model/model.tar.gz` |
| Model Card | `houselab/lab01/model/model_card.md` |
| Model Package | `house-price-predictor-lab01` v1, PendingManualApproval |

**Business outcome.** PropSmart's enquiry-to-quote path collapses from 3 days to a
sub-second model call, with an average error near \$26,000 — roughly 13% of mean
property value, and close to the noise floor of the data itself. Every prediction traces
back through the registry to a specific model version, its Model Card, its Feature
Group, and the exact catalogued CSV it was trained on. That end-to-end lineage, plus the
closed approval gate, is what makes the pipeline auditable as the executive mandate
requires.